In [1]:
import numpy as np
import pandas as pd
import river
import yfinance as yf
import datetime

In [2]:
ticker = 'AXISBANK.NS'
end_date = datetime.datetime.now()
start_date = end_date - datetime.timedelta(days=60)
data = yf.download(ticker, start=start_date, end=end_date)

[*********************100%***********************]  1 of 1 completed


In [3]:
data.head()

Price,Close,High,Low,Open,Volume
Ticker,AXISBANK.NS,AXISBANK.NS,AXISBANK.NS,AXISBANK.NS,AXISBANK.NS
Date,,,,,
2026-02-23,1386.699951,1402.000000,1380.199951,1380.5,6160153
2026-02-24,1387.599976,1398.000000,1383.500000,1398.0,7888492
2026-02-25,1403.000000,1404.300049,1387.599976,1394.0,4379373
2026-02-26,1395.500000,1405.699951,1386.000000,1403.0,8979547
2026-02-27,1383.900024,1395.500000,1381.000000,1387.0,5840317


In [5]:
data.describe()

Price,Close,High,Low,Open,Volume
Ticker,AXISBANK.NS,AXISBANK.NS,AXISBANK.NS,AXISBANK.NS,AXISBANK.NS
count,39.000000,39.000000,39.000000,39.000000,3.900000e+01
mean,1293.830770,1307.812829,1276.810259,1290.687174,7.520969e+06
std,76.680626,74.326191,78.630760,77.418083,2.814178e+06
min,1161.300049,1185.099976,1150.300049,1174.599976,2.744704e+06
25%,1218.399963,1236.750000,1200.849976,1214.599976,5.836335e+06
50%,1315.800049,1335.599976,1300.099976,1310.000000,6.789879e+06
75%,1356.899963,1371.700012,1349.700012,1360.150024,8.977199e+06
max,1403.000000,1405.699951,1387.599976,1403.000000,1.853489e+07


In [4]:
import numpy as np
import pandas as pd
import yfinance as yf

# =========================================
# CONFIG
# =========================================
STOCKS = {
    "HDFCBANK.NS": 0,
    "ICICIBANK.NS": 1,
    "SBIN.NS": 2,
    "AXISBANK.NS": 3,
    "KOTAKBANK.NS": 4
}

# =========================================
# FETCH DATA
# =========================================
def fetch_data(ticker, interval="1d", period="1y"):
    df = yf.download(
        ticker,
        interval=interval,
        period=period,
        auto_adjust=True,
        progress=False
    )

    df = df[['Open', 'High', 'Low', 'Close', 'Volume']]

    # 🔥 CRITICAL FIX: force everything to proper 1D numeric series
    df = df.apply(pd.to_numeric, errors='coerce')

    # 🔥 Flatten any accidental (n,1) columns
    df.columns = [col if not isinstance(col, tuple) else col[0] for col in df.columns]

    df = df.squeeze()  # <- THIS is the key line
    df = df.dropna()

    return df


def fetch_niftybank(interval="1d", period="1y"):
    nifty = yf.download(
        "^NSEBANK",
        interval=interval,
        period=period,
        auto_adjust=True,
        progress=False
    )

    nifty = nifty[['Close']]
    nifty = nifty.apply(pd.to_numeric, errors='coerce')

    nifty.columns = ['Close_nifty']
    nifty = nifty.squeeze()

    nifty = nifty.dropna()

    return nifty


# =========================================
# FEATURE ENGINEERING
# =========================================
def compute_features(df, nifty_df, stock_id, is_intraday=False):
    df = df.copy()

    # Align with Nifty Bank
    df = df.join(nifty_df, how='inner')

    # =========================
    # RETURNS
    # =========================
    df['log_return'] = np.log(df['Close'] / df['Close'].shift(1))

    df['ret_5d'] = np.log(df['Close'] / df['Close'].shift(5))
    df['ret_21d'] = np.log(df['Close'] / df['Close'].shift(21))
    df['ret_63d'] = np.log(df['Close'] / df['Close'].shift(63))

    df['nifty_ret_21d'] = np.log(df['Close_nifty'] / df['Close_nifty'].shift(21))
    df['rel_strength'] = df['ret_21d'] - df['nifty_ret_21d']

    # =========================
    # EMA RATIO
    # =========================
    ema_9 = df['Close'].ewm(span=9, adjust=False).mean()
    ema_50 = df['Close'].ewm(span=50, adjust=False).mean()
    df['ema_ratio'] = ema_9 / ema_50

    # =========================
    # RSI (14)


    delta = df['Close'].diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    roll_up = gain.rolling(14).mean()
    roll_down = loss.rolling(14).mean()

    rs = roll_up / (roll_down + 1e-9)
    df['rsi'] = 100 - (100 / (1 + rs))

    df['rsi'] = df['rsi'].clip(30, 70)

    # =========================
    # BOLLINGER WIDTH
    # =========================
    sma_20 = df['Close'].rolling(20).mean()
    std_20 = df['Close'].rolling(20).std()
    df['bb_width'] = (2 * std_20) / sma_20

    # =========================
    # ATR (14)
    # =========================
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift(1))
    low_close = np.abs(df['Low'] - df['Close'].shift(1))

    tr = np.maximum(high_low, np.maximum(high_close, low_close))
    df['atr_14'] = pd.Series(tr).rolling(14).mean()

    # =========================
    # VOLUME Z-SCORE
    # =========================
    vol_mean = df['Volume'].rolling(20).mean()
    vol_std = df['Volume'].rolling(20).std()
    df['vol_z'] = (df['Volume'] - vol_mean) / (vol_std + 1e-9)

    # =========================
    # VOLATILITY
    # =========================
    df['volatility_20d'] = df['log_return'].rolling(20).std()

    # =========================
    # INTRADAY FEATURES
    # =========================
    if is_intraday:
        df['cum_vol'] = df['Volume'].cumsum()
        df['cum_vol_price'] = (df['Close'] * df['Volume']).cumsum()

        df['vwap'] = df['cum_vol_price'] / (df['cum_vol'] + 1e-9)
        df['vwap_dev'] = (df['Close'] - df['vwap']) / df['vwap']

        df['time_index'] = np.arange(len(df)) / len(df)

    # =========================
    # META
    # =========================
    df['stock_id'] = stock_id

    # =========================
    # CLEAN
    # =========================
    df = df.drop(columns=['cum_vol', 'cum_vol_price'], errors='ignore')
    df = df.dropna()

    return df


# =========================================
# BUILD DATASET FOR ALL STOCKS
# =========================================
def build_dataset(interval="1d", period="1y", is_intraday=False):
    nifty_df = fetch_niftybank(interval=interval, period=period)

    all_data = []

    for ticker, stock_id in STOCKS.items():
        print(f"Processing {ticker}...")

        df = fetch_data(ticker, interval=interval, period=period)

        df_feat = compute_features(
            df,
            nifty_df=nifty_df,
            stock_id=stock_id,
            is_intraday=is_intraday
        )

        all_data.append(df_feat)

    final_df = pd.concat(all_data)
    final_df = final_df.sort_index()

    return final_df


# =========================================
# RUN
# =========================================
if __name__ == "__main__":
    
    # EOD DATA
    df_daily = build_dataset(interval="1d", period="1y", is_intraday=False)
    print("Daily dataset shape:", df_daily.shape)

    # INTRADAY DATA
    df_intraday = build_dataset(interval="30m", period="10d", is_intraday=True)
    print("Intraday dataset shape:", df_intraday.shape)

    # Split for ML
    X_daily = df_daily.drop(columns=['log_return'])
    y_daily = df_daily['log_return']

    X_intraday = df_intraday.drop(columns=['log_return'])
    y_intraday = df_intraday['log_return']

    print("Done.")

Processing HDFCBANK.NS...
Processing ICICIBANK.NS...
Processing SBIN.NS...
Processing AXISBANK.NS...
Processing KOTAKBANK.NS...
Daily dataset shape: (925, 19)
Processing HDFCBANK.NS...
Processing ICICIBANK.NS...
Processing SBIN.NS...
Processing AXISBANK.NS...
Processing KOTAKBANK.NS...
Intraday dataset shape: (275, 22)
Done.


Classification

In [5]:
# =========================================
# TARGET: DIRECTION
# =========================================
def prepare_classification(df):
    df = df.copy()

    # Binary target: 1 = up, 0 = down
    df['target'] = (df['log_return'] > 0).astype(int)

    # Drop original target
    X = df.drop(columns=['log_return', 'target'])
    y = df['target']

    return X, y

def time_series_split(df, split_ratio=0.8):
    df = df.sort_index()
    split_idx = int(len(df) * split_ratio)

    train = df.iloc[:split_idx]
    test = df.iloc[split_idx:]

    return train, test

from river import tree, metrics, drift

model = tree.HoeffdingAdaptiveTreeClassifier(
    grace_period=30,
    delta=1e-5
)

metric = metrics.Accuracy()
drift_detector = drift.ADWIN()

In [6]:
train_df, test_df = time_series_split(df_daily)

X_train, y_train = prepare_classification(train_df)
X_test, y_test = prepare_classification(test_df)

for x, y in zip(X_train.to_dict("records"), y_train):
    model.learn_one(x, y)

In [7]:
from collections import deque

metric = metrics.Accuracy()

# Optional: track probabilities
probs = []
preds = []

for x, y in zip(X_test.to_dict("records"), y_test):

    # Predict probability
    proba = model.predict_proba_one(x)
    prob_up = proba.get(1, 0.5)

    y_pred = 1 if prob_up > 0.5 else 0

    # Update metrics
    metric.update(y, y_pred)

    # Drift detection
    error = int(y_pred != y)
    drift_detector.update(error)

    if drift_detector.drift_detected:
        print("⚠️ Drift detected!")

    # Store for analysis
    probs.append(prob_up)
    preds.append(y_pred)

    # Learn
    model.learn_one(x, y)

print("Accuracy:", metric.get())

Accuracy: 0.5459459459459459


In [8]:
def generate_signal(prob_up):
    if prob_up > 0.6:
        return 1
    elif prob_up < 0.4:
        return 0
    else:
        return None
    
signals = [generate_signal(p) for p in probs]



In [9]:
print("Accuracy:", metric.get())

Accuracy: 0.5459459459459459


In [10]:
probs[:5]

[0.7256127371457267,
 0.35590305167872327,
 0.9645658207917726,
 0.03958100610625924,
 0.9999963702695139]

In [11]:
y_test

Date
2026-02-25    0
2026-02-25    1
2026-02-25    1
2026-02-25    0
2026-02-25    0
             ..
2026-04-23    0
2026-04-23    0
2026-04-23    0
2026-04-23    0
2026-04-23    0
Name: target, Length: 185, dtype: int64

In [12]:
filtered_correct = 0
filtered_total = 0

X_test_records = X_test.to_dict("records")

for p, y in zip(probs, y_test):

    signal = generate_signal(p)

    if signal is not None:
        filtered_total += 1
        if signal == y:
            filtered_correct += 1

print("Filtered accuracy:", filtered_correct / filtered_total)

Filtered accuracy: 0.5469613259668509


In [13]:
print("Total test samples:", len(y_test))
print("Signals generated:", filtered_total)

Total test samples: 185
Signals generated: 181


In [14]:
from sklearn.metrics import precision_score

filtered_preds = []
filtered_true = []

for p, y in zip(probs, y_test):

    signal = generate_signal(p)
    if signal is not None:
        filtered_preds.append(signal)
        filtered_true.append(y)

print("Precision:", precision_score(filtered_true, filtered_preds))

Precision: 0.49206349206349204


In [15]:
buy_correct = sell_correct = 0
buy_total = sell_total = 0

for p, y in zip(probs, y_test):

    signal = generate_signal(p)

    if signal is None:
        continue

    if signal == 1:
        buy_total += 1
        if y == 1:
            buy_correct += 1
    else:
        sell_total += 1
        if y == 0:
            sell_correct += 1

print("BUY precision:", buy_correct / buy_total)
print("SELL precision:", sell_correct / sell_total)

BUY precision: 0.49206349206349204
SELL precision: 0.576271186440678


In [85]:
print("BUY trades:", buy_total)
print("SELL trades:", sell_total)

BUY trades: 61
SELL trades: 109


In [16]:
def signal_A(prob_up):
    if prob_up < 0.3:
        return 0   # SELL
    else:
        return None
    
def signal_B(prob_up):
    if prob_up < 0.4:
        return 0   # SELL
    elif prob_up > 0.7:
        return 1   # strict BUY
    else:
        return None
    
def signal_C(prob_up):
    if prob_up < 0.4:
        return 0   # SELL
    elif prob_up > 0.6:
        return 1   # BUY
    else:
        return None
    
def evaluate_strategy(probs, y_test, signal_func):

    total = 0
    correct = 0

    preds = []
    true_vals = []

    buy_total = sell_total = 0
    buy_correct = sell_correct = 0

    for p, y in zip(probs, y_test):

        signal = signal_func(p)

        if signal is None:
            continue

        total += 1
        preds.append(signal)
        true_vals.append(y)

        if signal == y:
            correct += 1

        if signal == 1:
            buy_total += 1
            if y == 1:
                buy_correct += 1
        elif signal == 0:
            sell_total += 1
            if y == 0:
                sell_correct += 1

    accuracy = correct / total if total > 0 else 0

    from sklearn.metrics import precision_score
    precision = precision_score(true_vals, preds) if total > 0 else 0

    buy_precision = buy_correct / buy_total if buy_total > 0 else 0
    sell_precision = sell_correct / sell_total if sell_total > 0 else 0

    trade_ratio = total / len(y_test)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "buy_precision": buy_precision,
        "sell_precision": sell_precision,
        "trades": total,
        "trade_ratio": trade_ratio
    }

In [17]:
results_A = evaluate_strategy(probs, y_test, signal_A)
results_B = evaluate_strategy(probs, y_test, signal_B)
results_C = evaluate_strategy(probs, y_test, signal_C)

print("Strategy A (SELL only):", results_A)
print("Strategy B (SELL + strict BUY):", results_B)
print("Strategy C (balanced):", results_C)

Strategy A (SELL only): {'accuracy': 0.5789473684210527, 'precision': 0.0, 'buy_precision': 0, 'sell_precision': 0.5789473684210527, 'trades': 114, 'trade_ratio': 0.6162162162162163}
Strategy B (SELL + strict BUY): {'accuracy': 0.5568181818181818, 'precision': 0.5172413793103449, 'buy_precision': 0.5172413793103449, 'sell_precision': 0.576271186440678, 'trades': 176, 'trade_ratio': 0.9513513513513514}
Strategy C (balanced): {'accuracy': 0.5469613259668509, 'precision': 0.49206349206349204, 'buy_precision': 0.49206349206349204, 'sell_precision': 0.576271186440678, 'trades': 181, 'trade_ratio': 0.9783783783783784}


c:\Users\viole\stocks\stockenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [18]:
import pickle
MODEL_FILE = "dailymfinal.pkl"
with open(MODEL_FILE, "wb") as f:
    pickle.dump(model, f)